# Prepare SimBench 2016 Prosumer Data

Build the full-year SimBench prosumer CSV, then export quarterly segmented
train/test splits with warmup rows for forecast evaluation.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root


In [ ]:
from datasets.simbench_export import export_simbench_2016_dataset

price_candidates = [
    project_root / "data" / "Germany_price_15_2016.xlsx",
    project_root / "data" / "Germany_price_15_2016.csv",
]
price_path = next((path for path in price_candidates if path.exists()), None)
if price_path is None:
    raise FileNotFoundError(f"Missing Germany price source. Checked: {price_candidates}")

export_result, prosumer_rank_df = export_simbench_2016_dataset(
    output_dir=project_root / "data",
    price_path=price_path,
)

simbench_full_df = export_result.full_frame
simbench_train_df = export_result.train_frame
simbench_test_df = export_result.test_frame
metadata_payload = export_result.metadata

summary = {
    "price_path": str(price_path),
    "full_rows": len(simbench_full_df),
    "train_rows": len(simbench_train_df),
    "test_rows": len(simbench_test_df),
    "train_segments": metadata_payload["train_segments"],
    "test_segments": metadata_payload["test_segments"],
    "test_target_rows": metadata_payload["test_target_rows"],
    "test_warmup_rows": metadata_payload["test_warmup_rows"],
}
summary


In [ ]:
plot_steps = 96 * 7
plot_df = simbench_full_df.iloc[:plot_steps].copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
axes[0].plot(plot_df["timestamp"], plot_df["price"], label="price", color="black")
axes[0].set_ylabel("EUR/kWh")
axes[0].set_title("2016 Germany price (first week)")
axes[0].grid(True, linestyle=":")

for agent_idx in range(1, 4):
    axes[1].plot(plot_df["timestamp"], plot_df[f"load{agent_idx}"], label=f"load{agent_idx}")
axes[1].set_ylabel("kW")
axes[1].set_title("Selected prosumer loads (first week)")
axes[1].grid(True, linestyle=":")
axes[1].legend(loc="upper right")

for agent_idx in range(1, 4):
    axes[2].plot(plot_df["timestamp"], plot_df[f"pv{agent_idx}"], label=f"pv{agent_idx}")
axes[2].set_ylabel("kW")
axes[2].set_title("Selected prosumer PV outputs (first week)")
axes[2].grid(True, linestyle=":")
axes[2].legend(loc="upper right")

plt.tight_layout()
plt.show()

display(pd.DataFrame(metadata_payload["quarter_test_windows"]))
display(prosumer_rank_df.head(10))
display(simbench_train_df.head())
display(simbench_test_df.head())
